In [ ]:
import os
import numpy as np
import pandas as pd

RAW_DATA_PATH = os.path.join("..","data","raw","kaggle_dataset_temp.csv")
PROCESSED_DATA_DIR = os.path.join("..","data","processed")

df_raw = pd.read_csv(RAW_DATA_PATH)
print("Raw data loaded successfully.")
print(f"Initial shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

print("\n---Integrity Verification---")
missing_total = df_raw.isnull().sum().sum() # null per column + sum of all columns
duplicate_total = df_raw.duplicated().sum()
print(f"Total missing values: {missing_total}")
print(f"Total duplicate rows: {duplicate_total}")

assert missing_total == 0, "Warning: Missing values detected that require handling."
assert duplicate_total == 0, "Warning: Duplicate rows detected that require removal."

print("Data quality checks passed: Zero nulls and zero duplicates.")

Raw data loaded successfully.
Initial shape: 7308 rows, 7 columns

---Integrity Verification---
Total missing values: 0
Total duplicate rows: 0
Data quality checks passed: Zero nulls and zero duplicates.


In [5]:
# one-hot encoding (convert categorical data (location) into a binary format for machine learning)

df_clean = df_raw.copy()

location_encoded = pd.get_dummies(
    df_clean["Location"], prefix="Location", dtype=int
)

# concatenate horizontally (column)
df_clean = pd.concat([df_clean, location_encoded], axis=1)

print("--- Encoded Location Columns ---")
display(
    df_clean[["Location"] + [col for col in location_encoded.columns]].head()
)

--- Encoded Location Columns ---


,Location,Location_Manila,Location_Marikina,Location_Pasig,Location_Quezon City
0,Quezon City,0,0,0,1
1,Marikina,0,1,0,0
2,Manila,1,0,0,0
3,Pasig,0,0,1,0
4,Quezon City,0,0,0,1


In [ ]:
# reorder columns for clear structure: Metadata - Continuous Features - Static/Dummies - Targets
final_columns = [
    "Date",
    "Location",
    "Rainfall_mm",
    "WaterLevel_m",
    "SoilMoisture_pct",
    "Elevation_m",
    "Location_Manila",
    "Location_Marikina",
    "Location_Pasig",
    "Location_Quezon City",
    "FloodOccurrence",
]

# for exact naming matches
df_clean = df_clean[
    [col for col in final_columns if col in df_clean.columns]
]

print("--- Preprocessed DataFrame Summary ---")
display(df_clean.head())
print(f"Final column count: {df_clean.shape[1]}")

--- Preprocessed DataFrame Summary ---


,Date,Location,Rainfall_mm,WaterLevel_m,SoilMoisture_pct,Elevation_m,Location_Manila,Location_Marikina,Location_Pasig,Location_Quezon City,FloodOccurrence
0,2016-01-01,Quezon City,12.0,0.5,15.3,43,0,0,0,1,0
1,2016-01-01,Marikina,10.6,1.8,23.2,15,0,1,0,0,0
2,2016-01-01,Manila,5.7,0.5,15.6,5,1,0,0,0,0
3,2016-01-01,Pasig,3.7,0.5,5.0,5,0,0,1,0,0
4,2016-01-02,Quezon City,3.4,0.5,13.3,43,0,0,0,1,0


Final column count: 11


In [ ]:
# target summary and temporary target definition
# NOTE: Rainfall_mm serves as a temporary continuous target placeholder
# until archive research for FloodDepth_m is complete.

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
PROCESSED_FILE_PATH = os.path.join(PROCESSED_DATA_DIR, "cleaned_dataset.csv")

# saving a csv file to /processed 
df_clean.to_csv(PROCESSED_FILE_PATH, index=False)

print("Preprocessed dataset successfully saved.")
print(f"Destination: {PROCESSED_FILE_PATH}")
print(f"Final Exported Shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

df_verify = pd.read_csv(PROCESSED_FILE_PATH)
print("\n--- Verified Export Columns ---")
print(list(df_verify.columns))

Preprocessed dataset successfully saved.
Destination: ..\data\processed\cleaned_dataset.csv
Final Exported Shape: 7308 rows, 11 columns

--- Verified Export Columns ---
['Date', 'Location', 'Rainfall_mm', 'WaterLevel_m', 'SoilMoisture_pct', 'Elevation_m', 'FloodOccurrence', 'Location_Manila', 'Location_Marikina', 'Location_Pasig', 'Location_Quezon City']


## Preprocessing Summary & Target Specifications

### 1. Data Cleaning & Integrity
- **Raw File Source:** `data/raw/kaggle_dataset_temp.csv` (7,308 rows, 4 study cities across 2016–2020).
- **Quality Checks:** Zero missing values and zero duplicate records were identified, requiring no row deletion or imputation.

### 2. Feature Transformations
- **One-Hot Encoding:** Categorical `Location` was converted into binary indicator variables (`Location_Manila`, `Location_Marikina`, `Location_Pasig`, `Location_Quezon City`) to enable standard numeric matrix input for scikit-learn estimators.
- **Multicollinearity Context:** As demonstrated in EDA (Task 1.5b), `Elevation_m` is perfectly collinear with `Location` dummy variables. In subsequent modeling (Step 3), models will be evaluated with and without `Elevation_m` to prevent multicollinearity distortion in linear baselines.

### 3. Target Variable Definitions & Modeling Constraints
- **Regression Target Placeholder:** `Rainfall_mm` is currently designated as the continuous target placeholder until the LGU/news archive extraction of `FloodDepth_m` is fully populated.
- **Classification Target:** `RiskLevel` (MGB-derived tiers: Low <0.5m, Moderate 0.5–1.0m, High 1.0–2.0m) will be derived once `FloodDepth_m` is finalized.
- **Leakage Safeguard:** `FloodOccurrence` remains in the processed dataset for reference and classification baseline comparison, but will be excluded from feature matrices ($X$) during model training.

### 4. Output
- Exported clean master dataset to `data/processed/cleaned_dataset.csv`.